In [10]:
import numpy as np
import mne
from scipy.io import loadmat
from scipy.signal import welch
import matplotlib.pyplot as plt
from MFDFA import MFDFA # Make sure MFDFA library is installed (pip install MFDFA)
import glob
from scipy.stats import zscore # Added for zscore function

# 1. Load and Preprocess Data
def load_bci_data(file_path):
    """Load a single .mat file and return BCI structure."""
    try:
        mat = loadmat(file_path)
        bci = mat['BCI']
        # Handle cases where BCI might be a (1,1) numpy array wrapping the actual struct
        if isinstance(bci, np.ndarray) and bci.shape == (1, 1):
            bci = bci[0, 0]
        print(f"Loaded {file_path}: data shape={bci['data'].shape}, TrialData shape={bci['TrialData'].shape}")
        return bci
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def detect_artifacts(bci):
    """Detect bad channels (variance > 5z) and artifacted trials (>100 μV)."""
    try:
        sfreq = bci['SRATE'][0, 0]
        n_trials = bci['data'].shape[1]
        print(f"Processing {n_trials} trials with sfreq={sfreq}")
        bad_chan = []
        artifact_bool = np.zeros(n_trials, dtype=bool)
        max_amplitudes = []  # For histogram
        
        # Process each run
        runs = np.unique([bci['TrialData'][0, i]['runnumber'][0, 0] for i in range(n_trials)])
        print(f"Found {len(runs)} runs: {runs}")
        for run in runs:
            run_ind = [i for i in range(n_trials) if bci['TrialData'][0, i]['runnumber'][0, 0] == run]
            print(f"Run {run}: {len(run_ind)} trials")
            trials = bci['data'][0, run_ind]  # Shape: (n_run_trials, n_channels, n_samples)
            
            # Flatten trials for variance calculation
            trials_full = np.concatenate([t for t in trials], axis=1)  # Shape: (n_channels, total_samples)
            trials_full = trials_full.astype(np.float64)  # Cast to float64
            print(f"Run {run}: trials_full shape={trials_full.shape}")
            
            # Bandpass filter (8-30 Hz)
            trials_full_mne = mne.filter.filter_data(trials_full, sfreq, 8, 30, method='fir')
            
            # Detect bad channels
            elec_var = np.var(trials_full_mne, axis=1)
            elec_z = zscore(elec_var) # zscore is used here
            bad_chan_run = np.where(elec_z > 5)[0]
            bad_chan.extend(bad_chan_run)
            print(f"Run {run}: bad channels={bad_chan_run}")
            
            # Detect artifacts in trials
            for i, trial_idx in enumerate(run_ind):
                trial_data = trials[i].astype(np.float64)  # Cast to float64
                trial_data_mne = mne.filter.filter_data(trial_data, sfreq, 8, 30, method='fir')
                trial_data_mne[bad_chan_run, :] = 0  # Zero out bad channels
                # Check feedback period (4000 ms to resultind)
                resultind = bci['TrialData'][0, trial_idx]['resultind'][0, 0]
                feedback_start = int(4 * sfreq)
                feedback_end = int(resultind)
                if feedback_end > trial_data_mne.shape[1]:
                    print(f"Warning: trial {trial_idx + 1} resultind={resultind} exceeds data length={trial_data_mne.shape[1]}, clipping")
                    feedback_end = trial_data_mne.shape[1]
                feedback_data = trial_data_mne[:, feedback_start:feedback_end]
                max_ampl = np.max(np.abs(feedback_data))
                max_amplitudes.append(max_ampl * 1e6)  # Convert to μV
                artifact_bool[trial_idx] = max_ampl > 100e-6  # μV to V
        
        bad_chan = np.unique(bad_chan).tolist()
        return bad_chan, artifact_bool, max_amplitudes
    except Exception as e:
        print(f"Error in detect_artifacts: {e}")
        # Safely get n_trials for returning artifact_bool, in case bci['data'] is not accessible
        if isinstance(bci, dict) and 'data' in bci and bci['data'].shape[1] is not None:
            n_trials = bci['data'].shape[1]
        elif isinstance(bci, np.void) and 'data' in bci.dtype.fields and bci['data'].shape[1] is not None:
            n_trials = bci['data'].shape[1]
        else:
            # If bci is not in expected format, or 'data' key is missing, return empty array
            n_trials = 0 
        return [], np.zeros(n_trials, dtype=bool), []


def create_custom_montage(bci):
    """Create a custom DigMontage from chaninfo electrode positions."""
    try:
        electrodes = bci['chaninfo'][0, 0]['electrodes']
        ch_names = []
        pos_list = []
        for i, e in enumerate(electrodes[0]):  # Iterate over electrodes[0]
            label = e['label'][0] if isinstance(e['label'], np.ndarray) else e['label']
            if label == 'REF.':  # Skip reference channel
                continue
            # print(f"Electrode {i}: label={label}, X shape={e['X'].shape}, Y shape={e['Y'].shape}, Z shape={e['Z'].shape}")
            x = float(e['X'][0, 0])  # Extract scalar from (1,1) array
            y = float(e['Y'][0, 0])
            z = float(e['Z'][0, 0])
            ch_names.append(label)
            pos_list.append([x, y, z])
            
        pos = np.array(pos_list)  # Shape: (n_channels, 3)
        print(f"Created montage with {len(ch_names)} channels and position shape: {pos.shape}")
        if pos.shape[0] != len(ch_names):
            raise ValueError(f"Mismatch between channel names ({len(ch_names)}) and positions ({pos.shape[0]})")
            
        montage = mne.channels.make_dig_montage(
            ch_pos=dict(zip(ch_names, pos)),
            coord_frame='head'
        )
        return montage, ch_names
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error creating custom montage: {e}")
        return None, None

def create_mne_raw(bci, trial_idx=0):
    """Create MNE Raw object for a single trial."""
    montage, ch_names = create_custom_montage(bci)
    if montage is None or ch_names is None:
        raise ValueError("Failed to create custom montage")
        
    # Get EEG data
    try:
        eeg_data = bci['data'][0, trial_idx]
        print(f"Trial {trial_idx + 1}: eeg_data shape={eeg_data.shape}")
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
        if eeg_data.ndim != 2 or eeg_data.shape[0] != len(ch_names):
            raise ValueError(f"EEG data shape {eeg_data.shape} does not match n_channels={len(ch_names)}")
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing EEG data: {e}")
        raise
        
    sfreq = bci['SRATE'][0, 0]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data * 1e-6, info)  # Convert μV to V
    raw.set_montage(montage)
    
    # Filter for alpha band (8-14 Hz)
    raw.filter(8, 14, fir_design='firwin')
    
    # Mark bad channels (if present in BCI structure)
    try:
        # Check if 'noisechan' field exists and is not empty
        if 'noisechan' in bci['chaninfo'][0, 0].dtype.fields and len(bci['chaninfo'][0, 0]['noisechan'][0]) > 0:
            noisy_channels_indices = bci['chaninfo'][0, 0]['noisechan'][0]
            # Convert MATLAB 1-based indexing to Python 0-based
            bad_channels = [ch_names[int(i) - 1] for i in noisy_channels_indices] 
            raw.info['bads'] = bad_channels
            raw.interpolate_bads() # Interpolate bad channels
            print(f"Interpolated bad channels: {bad_channels}")
        else:
            print("No noisy channels found or 'noisechan' field is empty.")
    except (IndexError, KeyError) as e:
        print(f"Error accessing noisechan information: {e}")
        print("No noisy channels found or error accessing noisechan.")
    
    return raw

# 2. Visualizations
def plot_artifact_histogram(max_amplitudes, subject_id='S1'):
    """Plot histogram of maximum amplitudes (like Figure 1)."""
    plt.figure(figsize=(8, 5))
    plt.hist(max_amplitudes, bins=50, color='blue', alpha=0.7)
    plt.axvline(100, color='red', linestyle='--', label='100 μV threshold')
    plt.xlabel('Maximum Amplitude (μV)')
    plt.ylabel('Number of Trials')
    plt.title(f'Artifact Detection: Maximum Amplitudes for {subject_id}')
    plt.legend()
    plt.savefig(f'artifact_histogram_{subject_id}.png')
    plt.close()

def plot_time_series(raw, channels=['C3', 'C4']):
    """Plot EEG time series for selected channels."""
    try:
        # Check if channels exist in raw.info['ch_names']
        valid_channels = [ch for ch in channels if ch in raw.info['ch_names']]
        if not valid_channels:
            print(f"None of the specified channels {channels} found in raw data. Skipping time series plot.")
            return
        
        fig = raw.plot(scalings='auto', n_channels=len(valid_channels), show=False, block=False, picks=valid_channels)
        fig.suptitle('EEG Time Series (C3, C4)', y=0.98) # Use suptitle for overall title
        plt.savefig('eeg_time_series.png')
        plt.close(fig) # Close the figure to free memory
    except ValueError as e:
        print(f"Error plotting time series: {e}. Check if channels {channels} exist.")

def plot_psd(raw, channels=['C3', 'C4']):
    """Plot PSD for selected channels."""
    try:
        # Check if channels exist in raw.info['ch_names']
        valid_channels = [ch for ch in channels if ch in raw.info['ch_names']]
        if not valid_channels:
            print(f"None of the specified channels {channels} found in raw data. Skipping PSD plot.")
            return

        fig = raw.compute_psd(fmin=8, fmax=14, picks=valid_channels).plot(show=False)
        fig.suptitle('Power Spectral Density (8-14 Hz)', y=0.98) # Use suptitle for overall title
        plt.savefig('psd_c3_c4.png')
        plt.close(fig) # Close the figure to free memory
    except ValueError as e:
        print(f"Error plotting PSD: {e}. Check if channels {channels} exist.")

def plot_erd_ers(bci, trial_idx=0, task_number=None):
    """Plot ERD/ERS for C3/C4 during a trial, optionally filtered by task."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot plot ERD/ERS: Failed to create montage")
            return
        
        # Ensure C3 and C4 exist
        if 'C3' not in ch_names or 'C4' not in ch_names:
            print("C3 or C4 channel not found. Skipping ERD/ERS plot.")
            return
            
        c3_idx = ch_names.index('C3')
        c4_idx = ch_names.index('C4')
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
        
        # Check task number if provided
        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping ERD/ERS plot.")
                return
        
        eeg_c3 = eeg_data[c3_idx, :]
        eeg_c4 = eeg_data[c4_idx, :]
    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in plot_erd_ers (data access): {e}")
        return
        
    sfreq = bci['SRATE'][0, 0]
    try:
        resultind = bci['TrialData'][0, trial_idx]['resultind'][0, 0]
    except (IndexError, KeyError):
        print(f"Error accessing TrialData.resultind for trial {trial_idx}. Skipping ERD/ERS plot.")
        return
    
    # Define inter-trial (baseline) and feedback periods
    inter_trial = slice(0, int(2 * sfreq))  # -2000 to 0 ms relative to trial start
    feedback = slice(int(4 * sfreq), int(resultind))  # 4000 ms to resultind
    
    # Ensure slices are within bounds
    if inter_trial.stop > eeg_c3.shape[0] or feedback.stop > eeg_c3.shape[0]:
        print(f"Warning: Slices out of bounds for trial {trial_idx + 1}. Skipping ERD/ERS plot.")
        return
    
    # Compute Power Spectral Density (PSD) for alpha band (8-14 Hz)
    freqs, psd_c3_inter = welch(eeg_c3[inter_trial], fs=sfreq, nperseg=sfreq*1) # 1-sec segments
    _, psd_c3_feed = welch(eeg_c3[feedback], fs=sfreq, nperseg=sfreq*1)
    _, psd_c4_inter = welch(eeg_c4[inter_trial], fs=sfreq, nperseg=sfreq*1)
    _, psd_c4_feed = welch(eeg_c4[feedback], fs=sfreq, nperseg=sfreq*1)
    
    alpha_idx = (freqs >= 8) & (freqs <= 14)
    
    # Calculate ERD/ERS (Relative Power Change)
    mean_psd_c3_inter = np.mean(psd_c3_inter[:, alpha_idx]) if psd_c3_inter[:, alpha_idx].size > 0 else 1e-9 # Avoid division by zero
    mean_psd_c3_feed = np.mean(psd_c3_feed[:, alpha_idx]) if psd_c3_feed[:, alpha_idx].size > 0 else 0
    erd_c3 = (mean_psd_c3_feed - mean_psd_c3_inter) / mean_psd_c3_inter
    
    mean_psd_c4_inter = np.mean(psd_c4_inter[:, alpha_idx]) if psd_c4_inter[:, alpha_idx].size > 0 else 1e-9 # Avoid division by zero
    mean_psd_c4_feed = np.mean(psd_c4_feed[:, alpha_idx]) if psd_c4_feed[:, alpha_idx].size > 0 else 0
    erd_c4 = (mean_psd_c4_feed - mean_psd_c4_inter) / mean_psd_c4_inter
    
    plt.figure(figsize=(6, 4))
    plt.bar(['C3', 'C4'], [erd_c3, erd_c4], color=['red', 'blue'])
    plt.ylabel('ERD/ERS (Relative Power Change)')
    task_name = {1: 'LR', 2: 'UD', 3: '2D'}.get(task_number, f'Trial {trial_idx + 1}')
    plt.title(f'ERD/ERS for {task_name}')
    plt.savefig(f'erd_ers_task_{task_name}.png')
    plt.close()

def plot_topography(bci, trial_idx=0, task_number=None):
    """Plot topographic map of alpha power."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot plot topography: Failed to create montage")
            return
            
        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping topography plot.")
                return
        
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
            
    except (IndexError, KeyError, ValueError) as e:
        print(f"Error accessing EEG data for topography: {e}")
        return
        
    sfreq = bci['SRATE'][0, 0]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(eeg_data * 1e-6, info) # Convert to Volts
    raw.set_montage(montage)
    raw.filter(8, 14, fir_design='firwin') # Filter for alpha band
    
    # Compute PSD and then average alpha power
    psd, freqs = mne.time_frequency.psd_welch(raw, fmin=8, fmax=14, n_per_seg=raw.info['sfreq']*1) # 1-sec segments
    alpha_power = psd.mean(axis=1) # Average power across frequencies for each channel
    
    task_name = {1: 'LR', 2: 'UD', 3: '2D'}.get(task_number, f'Trial {trial_idx + 1}')
    
    fig, ax = plt.subplots(figsize=(6, 6)) # Create figure and axes explicitly
    mne.viz.plot_topomap(alpha_power, raw.info, show=False, axes=ax)
    ax.set_title(f'Alpha Power Topography ({task_name})')
    plt.savefig(f'topography_task_{task_name}.png')
    plt.close(fig)

def compute_pvc(bci, task_number=None, target_number=None):
    """Compute Percent Valid Correct (PVC) for a specific task and optional target."""
    try:
        n_trials = bci['TrialData'].shape[1]
        results = []
        for i in range(n_trials):
            task_match = task_number is None or bci['TrialData'][0, i]['tasknumber'][0, 0] == task_number
            target_match = target_number is None or bci['TrialData'][0, i]['targetnumber'][0, 0] == target_number
            if task_match and target_match:
                results.append(bci['TrialData'][0, i]['result'][0, 0])
        results = np.array(results)
        hits = np.sum(results == 1) # Assuming result == 1 means correct
        total = len(results)
        print(f"PVC for task={task_number}, target={target_number}: {hits}/{total} trials")
        return hits / total * 100 if total > 0 else 0
    except (IndexError, KeyError) as e:
        print(f"Error computing PVC: {e}")
        return None

def plot_pvc_across_subjects(data_path, task_number=1):
    """Plot PVC across subjects for a specific task (like Figure 2)."""
    pvc_data = []
    # Assuming subjects S1 to S20, adjust range if needed
    subject_ids = [f'S{i}' for i in range(1, 21)] 
    
    for subject_id in subject_ids:
        file_path = f'{data_path}/{subject_id}_Session_1.mat' # Assuming Session_1 is always relevant
        bci = load_bci_data(file_path)
        if bci is None:
            print(f"Could not load data for {subject_id}. Skipping.")
            continue
        pvc = compute_pvc(bci, task_number=task_number)
        if pvc is not None:
            pvc_data.append(pvc)
        else:
            pvc_data.append(0) # Append 0 if PVC could not be computed for consistency in plotting
            
    if pvc_data:
        # Filter out subjects that actually had data for plotting
        valid_subject_ids = [subject_ids[i] for i, pvc in enumerate(pvc_data) if pvc is not None]
        valid_pvc_data = [pvc for pvc in pvc_data if pvc is not None]

        plt.figure(figsize=(10, 6)) # Larger figure for readability
        plt.bar(valid_subject_ids, valid_pvc_data, color='#1e88e5')
        plt.axhline(70, color='red', linestyle='--', label='Proficiency Threshold (70%)')
        plt.xlabel('Subject')
        plt.ylabel('Percent Valid Correct (%)')
        task_map = {1: 'LR', 2: 'UD', 3: '2D'}
        plt.title(f'PVC for Task {task_map.get(task_number, "Unknown")} Across Subjects')
        plt.legend()
        plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout() # Adjust layout to prevent labels overlapping
        plt.savefig(f'pvc_task_{task_number}_subjects.png')
        plt.close()
    else:
        print(f"No valid PVC data found to plot for task {task_number} across subjects.")

def perform_mfdfa(bci, channel='C3', trial_idx=0, task_number=None):
    """Perform MFDFA on a single channel's EEG data."""
    try:
        montage, ch_names = create_custom_montage(bci)
        if montage is None or ch_names is None:
            print("Cannot perform MFDFA: Failed to create montage")
            return None, None, None
            
        if channel not in ch_names:
            print(f"Channel '{channel}' not found in data. Skipping MFDFA.")
            return None, None, None

        if task_number is not None:
            trial_task = bci['TrialData'][0, trial_idx]['tasknumber'][0, 0]
            if trial_task != task_number:
                print(f"Trial {trial_idx + 1} is task {trial_task}, not {task_number}. Skipping MFDFA.")
                return None, None, None
                
        ch_idx = ch_names.index(channel)
        eeg_data = bci['data'][0, trial_idx]
        if eeg_data.ndim == 1:
            n_channels = len(ch_names)
            n_samples = eeg_data.size // n_channels
            if n_samples * n_channels == eeg_data.size:
                eeg_data = eeg_data.reshape(n_channels, n_samples)
            else:
                raise ValueError(f"Cannot reshape 1D data of size {eeg_data.size} into (n_channels={n_channels}, n_samples)")
        elif eeg_data.ndim == 3 and eeg_data.shape[0] == 1:
            eeg_data = eeg_data[0]
            
        eeg_data_channel = eeg_data[ch_idx, :]
        
        if eeg_data_channel.size == 0:
            print(f"No data for channel {channel} in trial {trial_idx + 1}. Skipping MFDFA.")
            return None, None, None
            
    except (ValueError, KeyError, IndexError) as e:
        print(f"Error in perform_mfdfa (data access): {e}")
        return None, None, None
        
    # MFDFA parameters
    scales = np.arange(16, 2048, 32) # Define scales
    q = np.linspace(-5, 5, 101)    # Define q-moments
    
    try:
        lag, dfa = MFDFA(eeg_data_channel, scales, q, 1) # Order 1 polynomial detrending
    except Exception as e:
        print(f"Error during MFDFA computation for channel {channel}: {e}")
        return None, None, None

    h_q = []
    # Calculate generalized Hurst exponents (h(q))
    for i in range(len(q)):
        # Ensure there's enough data for polyfit
        if len(lag) > 1 and len(dfa[i, :]) > 1:
            coef = np.polyfit(np.log2(lag), np.log2(dfa[i, :]), 1)
            h_q.append(coef[0])
        else:
            h_q.append(np.nan) # Append NaN if not enough data

    h_q = np.array(h_q)
    
    # Calculate singularity strength (alpha) and multifractal spectrum (f(alpha))
    # Filter out NaNs if any, for gradient calculation
    valid_indices = ~np.isnan(h_q)
    if np.sum(valid_indices) > 1: # Need at least 2 valid points for gradient
        alpha_valid = h_q[valid_indices] + q[valid_indices] * np.gradient(h_q[valid_indices], q[valid_indices][1] - q[valid_indices][0])
        f_alpha_valid = q[valid_indices] * alpha_valid - (h_q[valid_indices] * q[valid_indices] - np.log2(dfa[valid_indices, -1]))
    else:
        print("Not enough valid data points to compute multifractal spectrum.")
        return None, None, None
    
    task_map = {1: 'LR', 2: 'UD', 3: '2D'}
    task_name = task_map.get(task_number, f'Trial {trial_idx + 1}')
    
    plt.figure(figsize=(6, 4))
    plt.plot(alpha_valid, f_alpha_valid, 'b-')
    plt.xlabel('Singularity Strength (α)')
    plt.ylabel('Multifractal Spectrum f(α)')
    plt.title(f'MFDFA Spectrum for {channel} ({task_name})')
    plt.grid(True)
    plt.savefig(f'mfdfa_spectrum_{task_name}.png')
    plt.close()
    
    return alpha_valid, f_alpha_valid, h_q

def compare_mbsr_control(data_path, task_number=1):
    """Compare PVC between MBSR and control groups for a task."""
    mbsr_pvc = []
    control_pvc = []
    # Assuming subjects S1 to S20, and 'metadata' containing 'MBSRsubject'
    for sub_num in range(1, 21): 
        file_path = f'{data_path}/S{sub_num}_Session_1.mat'
        bci = load_bci_data(file_path)
        if bci is None:
            print(f"Could not load data for S{sub_num}. Skipping.")
            continue
            
        pvc = compute_pvc(bci, task_number=task_number)
        if pvc is not None:
            try:
                # Check if 'metadata' and 'MBSRsubject' exist and get the value
                if 'metadata' in bci and 'MBSRsubject' in bci['metadata'][0, 0].dtype.fields:
                    is_mbsr = bci['metadata'][0, 0]['MBSRsubject'][0, 0] == 1
                    if is_mbsr:
                        mbsr_pvc.append(pvc)
                    else:
                        control_pvc.append(pvc)
                else:
                    print(f"MBSR subject metadata not found for S{sub_num}. Skipping classification.")
            except (KeyError, IndexError) as e:
                print(f"Error accessing MBSRsubject for subject S{sub_num}: {e}")
    
    task_map = {1: 'LR', 2: 'UD', 3: '2D'}
    if mbsr_pvc and control_pvc:
        print(f"Task {task_number} ({task_map.get(task_number, 'Unknown')}):")
        print(f"MBSR group PVC: {np.mean(mbsr_pvc):.2f} ± {np.std(mbsr_pvc):.2f} (n={len(mbsr_pvc)})")
        print(f"Control group PVC: {np.mean(control_pvc):.2f} ± {np.std(control_pvc):.2f} (n={len(control_pvc)})")
        
        plt.figure(figsize=(6, 5))
        plt.bar(['MBSR', 'Control'], [np.mean(mbsr_pvc), np.mean(control_pvc)], 
                yerr=[np.std(mbsr_pvc), np.std(control_pvc)], color=['#1e88e5', '#d81b60'], capsize=5)
        plt.ylabel('Percent Valid Correct (%)')
        plt.title(f'PVC Comparison for Task {task_map.get(task_number, "Unknown")}')
        plt.ylim(0, 100) # Ensure y-axis goes from 0 to 100
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(f'pvc_mbsr_vs_control_task_{task_number}.png')
        plt.close()
    else:
        print(f"Not enough valid PVC data for task {task_number} to compare MBSR and Control groups.")


# Main execution block
if __name__ == '__main__':
    data_path = '.'  # Set this to the directory where your .mat files are located
    subject_id = 'S1' # Example subject ID
    
    try:
        print(f"--- Loading data for {subject_id} ---")
        file_path = f'{data_path}/{subject_id}_Session_1.mat'
        bci = load_bci_data(file_path)
        if bci is None:
            raise ValueError(f"Failed to load {file_path}. Please ensure the file exists and is accessible.")
        
        print("\n--- Detecting Artifacts ---")
        bad_chan, artifact_bool, max_amplitudes = detect_artifacts(bci)
        print(f"Bad channels detected: {bad_chan}")
        print(f"Number of artifacted trials: {np.sum(artifact_bool)} out of {len(artifact_bool)}")
        
        # Plot Artifact Histogram
        plot_artifact_histogram(max_amplitudes, subject_id)
        
        # Filter for valid, correct, and non-artifacted trials
        valid_correct_non_artifact_trials = [
            i for i in range(bci['TrialData'].shape[1]) 
            if not artifact_bool[i] and bci['TrialData'][0, i]['result'][0, 0] == 1
        ]
        
        if not valid_correct_non_artifact_trials:
            print("\nNo valid, correct, non-artifacted trials found for plotting detailed trial-specific visualizations.")
        else:
            # Use the first available valid trial for general visualizations
            first_valid_trial_idx = valid_correct_non_artifact_trials[0]
            print(f"\n--- Creating MNE Raw object for a valid trial (index {first_valid_trial_idx + 1}) ---")
            raw_for_general_plots = create_mne_raw(bci, trial_idx=first_valid_trial_idx)
            
            print("\n--- Generating General Visualizations (Time Series, PSD) ---")
            plot_time_series(raw_for_general_plots, channels=['C3', 'C4'])
            plot_psd(raw_for_general_plots, channels=['C3', 'C4'])

            print("\n--- Generating Task-Specific Visualizations (ERD/ERS, Topography, MFDFA) ---")
            for task_num in [1, 2, 3]: # Task numbers for LR, UD, 2D
                task_trials = [i for i in valid_correct_non_artifact_trials if bci['TrialData'][0, i]['tasknumber'][0, 0] == task_num]
                if task_trials:
                    # Use the first available valid trial for the current task
                    trial_for_task = task_trials[0]
                    print(f"Generating plots for Task {task_num} using trial {trial_for_task + 1}")
                    plot_erd_ers(bci, trial_idx=trial_for_task, task_number=task_num)
                    plot_topography(bci, trial_idx=trial_for_task, task_number=task_num)
                    perform_mfdfa(bci, channel='C3', trial_idx=trial_for_task, task_number=task_num)
                else:
                    print(f"No valid, correct, non-artifacted trials found for Task {task_num}. Skipping task-specific plots.")

        print("\n--- Plotting PVC Across Subjects ---")
        for task_num in [1, 2, 3]: # Task numbers for LR, UD, 2D
            plot_pvc_across_subjects(data_path, task_number=task_num)
            
        print("\n--- Comparing MBSR vs. Control Group PVC ---")
        for task_num in [1, 2, 3]: # Task numbers for LR, UD, 2D
            compare_mbsr_control(data_path, task_number=task_num)
            
        print("\nAnalysis complete. Check the script's directory for generated PNG images.")

    except Exception as e:
        print(f"\nAn error occurred during main execution: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for debugging

--- Loading data for S1 ---
Loaded ./S1_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)

--- Detecting Artifacts ---
Processing 450 trials with sfreq=1000
Found 18 runs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
Run 1: 25 trials
Run 1: trials_full shape=(62, 256344)
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


Run 1: bad channels=[]
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff fre

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s



FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting

...

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s


- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 1651 samples (1.651 s)

Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband atte

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s



Number of artifacted trials: 450 out of 450

No valid, correct, non-artifacted trials found for plotting detailed trial-specific visualizations.

--- Plotting PVC Across Subjects ---
Loaded ./S1_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 55/150 trials
Loaded ./S2_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 71/150 trials
Loaded ./S3_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 32/150 trials
Loaded ./S4_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 63/150 trials
Loaded ./S5_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 103/150 trials
Loaded ./S6_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 48/150 trials
Loaded ./S7_Session_1.mat: data shape=(1, 450), TrialData shape=(1, 450)
PVC for task=1, target=None: 59/150 trials
Loa

Traceback (most recent call last):
  File "C:\Users\Amar\AppData\Local\Temp\ipykernel_324\1300449286.py", line 591, in <module>
    compare_mbsr_control(data_path, task_number=task_num)
  File "C:\Users\Amar\AppData\Local\Temp\ipykernel_324\1300449286.py", line 504, in compare_mbsr_control
    if 'metadata' in bci and 'MBSRsubject' in bci['metadata'][0, 0].dtype.fields:
       ^^^^^^^^^^^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
